In [ ]:
from __future__ import annotations

# ---------------------------------------------------------------------
# IMPORTANT: cap BLAS thread count BEFORE numpy / sklearn are imported.
# When the GA training loop runs through joblib's loky backend (one
# process per seed), each worker still gets its own BLAS thread pool;
# without this cap, BLAS would oversubscribe cores and tank the speedup.
# os.environ.setdefault preserves any externally-set override.
# ---------------------------------------------------------------------
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS",      "1")
os.environ.setdefault("OMP_NUM_THREADS",      "1")
os.environ.setdefault("BLIS_NUM_THREADS",     "1")

from deap import creator, base
# Define the DEAP creator classes in THIS (parent) process so worker
# results -- which contain creator.Individual / creator.IndividualSingle
# instances -- can be unpickled here. Each worker also recreates them
# lazily inside MultiObjectiveTraining.run / SingleObjectiveTraining.run
# (those guards are idempotent: `if name not in creator.__dict__`).
if "FitnessMulti" not in creator.__dict__:
    creator.create("FitnessMulti", base.Fitness, weights=(1.0, 1.0))
if "Individual" not in creator.__dict__:
    creator.create("Individual", list, fitness=creator.FitnessMulti)
if "FitnessSingle" not in creator.__dict__:
    creator.create("FitnessSingle", base.Fitness, weights=(1.0,))
if "IndividualSingle" not in creator.__dict__:
    creator.create("IndividualSingle", list, fitness=creator.FitnessSingle)

import numpy
import random
import pandas
import time
from scipy.stats import pointbiserialr
import seaborn as sns
from typing import Any
from joblib import Parallel, delayed
from sklearn.metrics import average_precision_score, matthews_corrcoef, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

from training_config import TrainingConfig
from training_utils import select_pareto_individual
from multi_objective_training import MultiObjectiveTraining
from single_objective_training import SingleObjectiveTraining
from forward_stepwise_training import ForwardStepwiseTraining
from evaluation_utils import ensure_directory as _ensure_directory
from evaluation_utils import (
    get_continuous_columns, get_dummy_columns,
    build_model_package, evaluate_and_save,
    compute_marginal_correlations,
)
from evaluation_utils import apply_proportional_noise, apply_dummy_noise, evaluate_model

In [ ]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    numpy.random.seed(seed)

In [ ]:
CSV_TRAIN_PATH: str = "arrhythmia/arrhythmia_preprocessed_train_data.csv"
CSV_TEST_PATH: str = "arrhythmia/arrhythmia_preprocessed_test_data.csv"
TARGET_COLUMN: str = "Outcome"

#CSV_TRAIN_PATH: str = "readmit/readmit_130_hospitals_preprocessed_train_data.csv"
#CSV_TEST_PATH: str = "readmit/readmit_130_hospitals_preprocessed_test_data.csv"
#TARGET_COLUMN: str = "target_readmitted"

RESULT_PATH: str = time.strftime("%Y-%m-%d_%H-%M-%S")
RESULT_PATH_MULTI: str = f"{RESULT_PATH}\\multi"
RESULT_PATH_SINGLE: str = f"{RESULT_PATH}\\single"
RESULT_PATH_EVAL: str = f"{RESULT_PATH}\\evaluation"

# MORSE pareto front model selection based on knee point algorithm or select the best sign consistency score.
USE_KNEE_POINT_SELECTION: bool = False

# Using ROC-AUC as a main objective function or use PR-AUC instead.
USE_ROC_AUC: bool = False
MAIN_OBJECTIVE: str = "ROC-AUC" if USE_ROC_AUC == True else "PR-AUC"

# Parallelism for the GA training loop.
#   -1 = use all available CPU cores (one worker process per seed, up to that many in flight).
N_JOBS: int = -1

In [ ]:
# Load train dataset.
df_train: pandas.DataFrame = pandas.read_csv(CSV_TRAIN_PATH)

# Split data into training and validation sets.
X_search_pandas: pandas.DataFrame = df_train.drop(columns=[TARGET_COLUMN])
y_search_pandas: pandas.Series = df_train[TARGET_COLUMN]

X_search: numpy.ndarray = numpy.ascontiguousarray(X_search_pandas.to_numpy(), dtype=numpy.float64)
y_search: numpy.ndarray = numpy.ascontiguousarray(y_search_pandas.to_numpy(), dtype=numpy.float64)

# Store feature names.
feature_names: list[str] = list(X_search_pandas.columns)

In [ ]:
# Load test dataset.
df_test: pandas.DataFrame = pandas.read_csv(CSV_TEST_PATH)

y_test: pandas.Series = df_test[TARGET_COLUMN]
X_test: pandas.DataFrame = df_test.drop(columns=[TARGET_COLUMN])

In [ ]:
# Cross-validation splitter and full-train marginal correlations.
cv: StratifiedKFold = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Full-train marginal correlations, aligned with feature_names order.
corr_array: numpy.ndarray = compute_marginal_correlations(X_search, y_search)

In [ ]:
training_results_multi: dict[int, list[list[creator.Individual]]] = {}
training_results_single: dict[int, list[creator.IndividualSingle]] = {}
training_results_fwd: dict[int, list[int]] = {}

# 20 seeds fit training, deterministic ordering keeps results comparable across runs.
seeds: list[int] = list(range(42, 62))

# Inner SFS parallelism: 1 when the outer seed pool is parallel
# (avoid loky x sklearn oversubscription); -1 when running sequentially.
_sfs_inner_n_jobs: int = 1 if N_JOBS != 1 else -1

def _train_one_seed(s: int):
    """
    Run MORSE + SOGA + SFS end-to-end for one seed and return the results.
    """
    _seed_start: float = time.time()
    print(f"  [seed {s:>4d}] starting MORSE + SOGA + SFS pipeline...", flush=True)

    # --- MORSE ---
    set_seed(s)
    mo: MultiObjectiveTraining = MultiObjectiveTraining(
        config=TrainingConfig(seed=s, result_directory=RESULT_PATH_MULTI, use_roc_auc=USE_ROC_AUC),
        feature_names=feature_names,
        X_train=X_search,
        y_train=y_search,
        cv=cv,
        corr_array=corr_array,
    )
    pareto_front: list[creator.Individual] = mo.run()
    mo.clear_cache()
    _morse_dt: float = time.time() - _seed_start
    print(f"  [seed {s:>4d}] MORSE done after {_morse_dt/60:5.1f} min "
          f"({len(pareto_front)} Pareto inds)", flush=True)

    # --- SOGA ---
    set_seed(s)
    _soga_start: float = time.time()
    so: SingleObjectiveTraining = SingleObjectiveTraining(
        config=TrainingConfig(seed=s, result_directory=RESULT_PATH_SINGLE, use_roc_auc=USE_ROC_AUC),
        feature_names=feature_names,
        X_train=X_search,
        y_train=y_search,
        cv=cv,
    )
    best_indi: creator.IndividualSingle = so.run()
    so.clear_cache()
    _soga_dt: float = time.time() - _soga_start
    print(f"  [seed {s:>4d}] SOGA done after {_soga_dt/60:5.1f} min", flush=True)

    # --- SFS ---
    set_seed(s)
    _sfs_start: float = time.time()
    sfs: ForwardStepwiseTraining = ForwardStepwiseTraining(
        config=TrainingConfig(seed=s, result_directory="", use_roc_auc=USE_ROC_AUC),
        X_train=X_train_df.to_numpy(),
        y_train=y_train_series,
        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=s),
        inner_n_jobs=_sfs_inner_n_jobs,
    )
    fwd_individual: list[int] = sfs.run()
    _sfs_dt: float = time.time() - _sfs_start
    print(f"  [seed {s:>4d}] SFS done after {_sfs_dt/60:5.1f} min", flush=True)

    _total_dt: float = time.time() - _seed_start
    print(f"  [seed {s:>4d}] all three methods done -- seed total {_total_dt/60:5.1f} min",
          flush=True)

    return s, pareto_front, best_indi, fwd_individual


# Seed-level parallelism. Each worker process runs one full
# (MORSE + SOGA + SFS) pipeline for a single seed.
_overall_start: float = time.time()
print(f"Training {len(seeds)} seeds in parallel (N_JOBS={N_JOBS}, backend=loky)...",
      flush=True)

_results: list = []
if N_JOBS == 1:
    for _i, _s in enumerate(seeds, 1):
        _results.append(_train_one_seed(_s))
        _el: float = time.time() - _overall_start
        print(f"Progress: {_i}/{len(seeds)} seeds done | "
              f"elapsed {_el/60:6.1f} min | avg {_el/_i/60:5.1f} min/seed",
              flush=True)
else:
    _parallel = Parallel(n_jobs=N_JOBS, backend="loky", return_as="generator")
    _gen = _parallel(delayed(_train_one_seed)(s) for s in seeds)
    for _i, _r in enumerate(_gen, 1):
        _results.append(_r)
        _el = time.time() - _overall_start
        _eta = _el / _i * (len(seeds) - _i)
        print(f"Progress: {_i}/{len(seeds)} seeds done | "
              f"elapsed {_el/60:6.1f} min | avg {_el/_i/60:5.1f} min/seed | "
              f"ETA ~{_eta/60:5.1f} min",
              flush=True)

for s, pareto_front, best_indi, fwd_individual in _results:
    training_results_multi[s] = pareto_front
    training_results_single[s] = best_indi
    training_results_fwd[s] = fwd_individual

_total_min: float = (time.time() - _overall_start) / 60.0
print(f"All {len(seeds)} seeds finished in {_total_min:.1f} minutes.", flush=True)

## Baseline: Logistic Regression with ALL features (no feature selection)

Train a logistic regression using **every** available feature (no GA, no stepwise search) and report clean AUC/AP on the test set per seed. Serves as the trivial "no selection" reference point against which both the GA-based methods and the forward-stepwise baseline must justify themselves.

In [ ]:
# Baseline 1: Logistic Regression using ALL features (no feature selection)
baseline_all_rows: list[dict] = []
all_features_ind: list[int] = [1] * len(feature_names)

print("=== Baseline: Logistic Regression with ALL features ===\n")
for s in seeds:
    set_seed(s)

    model_pkg_all: dict[str, Any] = build_model_package(
        all_features_ind, feature_names, X_train_df, y_train_series, seed=s)

    X_test_scaled_all: numpy.ndarray = model_pkg_all["scaler"].transform(
        X_test[model_pkg_all["features"]].to_numpy())
    y_prob_all: numpy.ndarray = model_pkg_all["model"].predict_proba(X_test_scaled_all)[:, 1]

    if USE_ROC_AUC:
        test_auc_all: float = float(roc_auc_score(y_test, y_prob_all))
    else:
        test_auc_all: float = float(average_precision_score(y_test, y_prob_all))

    baseline_all_rows.append({
        "seed": s,
        "n_features": len(model_pkg_all["features"]),
        "test_auc": test_auc_all,
    })
    print(f"  Seed {s} | Features: {len(model_pkg_all['features']):4d} "
          f"| Test AUC: {test_auc_all:.4f}")

baseline_all_df: pandas.DataFrame = pandas.DataFrame(baseline_all_rows)
print(f"\nMean test AUC: {baseline_all_df['test_auc'].mean():.4f} "
      f"(+/- {baseline_all_df['test_auc'].std():.4f})")

baseline_all_dir: str = os.path.join(RESULT_PATH_EVAL, "baseline_all_features")
_ensure_directory(baseline_all_dir)
baseline_all_df.to_csv(
    os.path.join(baseline_all_dir, "baseline_all_features.csv"), index=False)
print(f"\nBaseline (all features) results saved to: {baseline_all_dir}")

## All-Models Noise Robustness Comparison (averaged across seeds)

Evaluate all four models on the **test set** under two noise sweeps:
- **Gaussian noise** on continuous features (zero mean-shift), noise level = 0.0 .. 1.0 in 0.1 steps.
- **Random corruption** on binary (dummy) features, corruption fraction = 0.0 .. 1.0 in 0.1 steps.

For each seed the same four models are evaluated at every noise level, and the mean AUC across seeds is plotted with an std shaded band. This gives a like-for-like view of robustness for:
- **Multi-Objective (MORSE)** Pareto individual with max sign-consistency or knee point
- **Single-Objective (AUC-only GA)** best HoF individual
- **All features (no selection)** logistic regression on every input
- **Forward stepwise selection** `SequentialFeatureSelector` (forward, CV=3, PR-AUC or ROC-AUC, tol=1e-3)

The two output plots match the style of the per-seed `gaussian_noise_comparison_test.png` and `dummy_flip_comparison_test.png` files, but with four curves and seed-aggregated statistics.

In [ ]:
# All-models noise robustness comparison (averaged across seeds)
noise_levels: numpy.ndarray = numpy.arange(0.0, 1.1, 0.1)
flip_levels: numpy.ndarray = numpy.arange(0.0, 1.05, 0.1)

gauss_rows: list[dict] = []
dummy_rows: list[dict] = []

all_features_ind: list[int] = [1] * len(feature_names)

print("Running noise sweeps for all 4 models across all seeds...")
for s in seeds:
    set_seed(s)

    # --- Build all 4 model packages on the same full training set ---
    pareto_front: list[creator.Individual] = training_results_multi[s]
    best_multi_ind: creator.Individual = select_pareto_individual(pareto_front, use_knee_point=USE_KNEE_POINT_SELECTION)

    pkg_multi: dict[str, Any] = build_model_package(
        best_multi_ind, feature_names, X_train_df, y_train_series, seed=s)
    pkg_single: dict[str, Any] = build_model_package(
        training_results_single[s], feature_names, X_train_df, y_train_series, seed=s)
    pkg_all: dict[str, Any] = build_model_package(
        all_features_ind, feature_names, X_train_df, y_train_series, seed=s)

    # Forward stepwise: reuse the per-seed selection cached by the
    # forward-stepwise baseline cell (cell 22). Avoids re-fitting SFS
    # here, which used to be the dominant cost of this cell.
    fwd_individual: list[int] = baseline_fwd_individuals_by_seed[s]
    pkg_fwd = build_model_package(
        fwd_individual, feature_names, X_train_df, y_train_series, seed=s)

    models: dict[str, dict] = {
        "multi":   pkg_multi,
        "single":  pkg_single,
        "all":     pkg_all,
        "forward": pkg_fwd,
    }

    # --- Gaussian noise sweep (zero mean-shift) ---
    for noise in noise_levels:
        nv: float = round(float(noise), 2)
        X_noisy: pandas.DataFrame = apply_proportional_noise(
            X_test, train_std, nv, 0.0, continuous_cols)
        row: dict = {"seed": s, "noise_level": nv}
        for label, pkg in models.items():
            row[f"auc_{label}"] = evaluate_model(pkg, X_noisy, y_test, use_roc_auc=USE_ROC_AUC)
        gauss_rows.append(row)

    # --- Dummy corruption sweep ---
    for flip in flip_levels:
        fv: float = round(float(flip), 2)
        X_noisy = apply_dummy_noise(X_test, fv, dummy_cols)
        row = {"seed": s, "flip_rate": fv}
        for label, pkg in models.items():
            row[f"auc_{label}"] = evaluate_model(pkg, X_noisy, y_test, use_roc_auc=USE_ROC_AUC)
        dummy_rows.append(row)

    print(f"  Seed {s} done "
          f"(features: multi={sum(best_multi_ind)}, single={sum(training_results_single[s])}, "
          f"all={len(feature_names)}, forward={sum(fwd_individual)})")

gauss_df: pandas.DataFrame = pandas.DataFrame(gauss_rows)
dummy_df: pandas.DataFrame = pandas.DataFrame(dummy_rows)

# Aggregate (mean, std) across seeds
gauss_agg: pandas.DataFrame = gauss_df.drop(columns=["seed"]).groupby("noise_level").agg(["mean", "std"])
dummy_agg: pandas.DataFrame = dummy_df.drop(columns=["seed"]).groupby("flip_rate").agg(["mean", "std"])

comparison_dir: str = os.path.join(RESULT_PATH_EVAL, "all_models_comparison")
_ensure_directory(comparison_dir)
gauss_df.to_csv(os.path.join(comparison_dir, "gaussian_noise_per_seed.csv"), index=False)
dummy_df.to_csv(os.path.join(comparison_dir, "dummy_flip_per_seed.csv"), index=False)

# Plot style: one entry per model (label, color, marker, linestyle)
model_styles: dict[str, tuple] = {
    "multi":   ("Multi-Objective (MORSE)",    "tab:blue",   "o", "-"),
    "single":  ("Single-Objective (AUC-only GA)", "tab:orange", "s", "--"),
    "all":     ("All features (no selection)",    "tab:green",  "^", "-."),
    "forward": ("Forward stepwise selection",     "tab:red",    "D", ":"),
}


def _plot_comparison(agg_df: pandas.DataFrame, x_col_name: str,
                     xlabel: str, title: str, out_path: str) -> None:
    fig, ax = plt.subplots(figsize=(10, 6))
    for key, (name, color, marker, ls) in model_styles.items():
        mean_series: pandas.Series = agg_df[(f"auc_{key}", "mean")]
        std_series: pandas.Series = agg_df[(f"auc_{key}", "std")].fillna(0.0)
        x_vals: numpy.ndarray = mean_series.index.values
        m_vals: numpy.ndarray = mean_series.values
        s_vals: numpy.ndarray = std_series.values
        ax.plot(x_vals, m_vals, label=name,
                color=color, marker=marker, linestyle=ls, linewidth=2)
        ax.fill_between(x_vals, m_vals - s_vals, m_vals + s_vals,
                        color=color, alpha=0.15)
    ax.set_xlabel(xlabel, fontweight="bold")
    ax.set_ylabel(f"Test {MAIN_OBJECTIVE} (mean across seeds; shaded = +/- 1 std)", fontweight="bold")
    ax.set_title(title, fontweight="bold", pad=10)
    ax.legend(frameon=True, fancybox=True, shadow=True)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


_plot_comparison(
    gauss_agg, "noise_level",
    "Gaussian Noise Level (fraction of training std, no mean shift)",
    "Robustness on Test Set: Additive Gaussian Noise on Continuous Features\n"
    "(all 4 models, averaged across seeds)",
    os.path.join(comparison_dir, "gaussian_noise_comparison_test.png"))

_plot_comparison(
    dummy_agg, "flip_rate",
    "Corruption Fraction (proportion of dummy cells randomised)",
    "Robustness on Test Set: Random Corruption Noise on Binary (Dummy) Features\n"
    "(all 4 models, averaged across seeds)",
    os.path.join(comparison_dir, "dummy_flip_comparison_test.png"))

print(f"\nAll-models comparison plots and CSVs saved to: {comparison_dir}")

## Feature Count Comparison Across Seeds

For each method (MORSE, SOGA, all-features, forward-stepwise) plot the number of selected features per seed.

- MORSE and SOGA vary with the random seed (different GA inits / CV shuffles produce different Pareto fronts and HoFs).
- "All features" is by construction constant at `n_features`.
- Forward stepwise selects its own number of features per seed via SFS' `tol`-based auto stopping (no matched budget; pre-cached in cell 22).

The plot is a boxplot (quartile summary) with individual seeds overlaid as jittered dots, so both the distribution shape and the per-seed values are visible. Depends on `best_multi_by_seed`, `best_single_by_seed` (from the stability cell) and `baseline_fwd_df` (from the forward-stepwise baseline cell).

In [ ]:
# Feature count comparison across seeds
feature_count_rows: list[dict] = []
for s in seeds:
    feature_count_rows.append({
        "method": "MORSE",
        "seed": s,
        "n_features": int(sum(best_multi_by_seed[s])),
    })
    feature_count_rows.append({
        "method": "SOGA (AUC)",
        "seed": s,
        "n_features": int(sum(best_single_by_seed[s])),
    })
    feature_count_rows.append({
        "method": "All features",
        "seed": s,
        "n_features": int(len(feature_names)),
    })

# Forward stepwise: pull directly from the per-seed baseline table.
for _, _row in baseline_fwd_df.iterrows():
    feature_count_rows.append({
        "method": "Forward stepwise",
        "seed": int(_row["seed"]),
        "n_features": int(_row["n_features"]),
    })

feature_count_df: pandas.DataFrame = pandas.DataFrame(feature_count_rows)

print("=== Feature Count Comparison Across Seeds ===\n")
summary_df: pandas.DataFrame = (
    feature_count_df.groupby("method")["n_features"]
    .agg(["min", "median", "mean", "max", "std"])
    .round(2)
)
print(summary_df.to_string())

# --- Plot: boxplot + stripplot overlay ---
method_order: list[str] = ["MORSE", "SOGA (AUC)", "Forward stepwise", "All features"]
palette: dict[str, str] = {
    "MORSE":      "tab:blue",
    "SOGA (AUC)":       "tab:orange",
    "Forward stepwise": "tab:red",
    "All features":     "tab:green",
}

fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(
    data=feature_count_df,
    x="method", y="n_features",
    order=method_order, hue="method", palette=palette,
    legend=False, width=0.5, ax=ax,
)
sns.stripplot(
    data=feature_count_df,
    x="method", y="n_features",
    order=method_order,
    color="black", size=4, alpha=0.6, jitter=0.15, ax=ax,
)

# Annotate each box with the median value
for i, method in enumerate(method_order):
    med: float = feature_count_df.loc[feature_count_df["method"] == method, "n_features"].median()
    ax.annotate(f"median={int(med)}",
                xy=(i, med), xytext=(0, 14),
                textcoords="offset points", ha="center",
                fontsize=9, color="black",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="gray", alpha=0.8))

ax.set_xlabel("Method", fontweight="bold")
ax.set_ylabel(f"Number of selected features (out of {len(feature_names)})", fontweight="bold")
ax.set_title(f"Feature Count per Method Across {len(seeds)} Seeds\n"
             f"(box = IQR with median line, dots = individual seeds)",
             fontweight="bold", pad=10)
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()

feature_count_dir: str = os.path.join(RESULT_PATH_EVAL, "feature_counts")
_ensure_directory(feature_count_dir)
fig.savefig(os.path.join(feature_count_dir, "feature_count_comparison.png"), dpi=150)
plt.close(fig)

feature_count_df.to_csv(
    os.path.join(feature_count_dir, "feature_count_per_seed.csv"), index=False)
summary_df.to_csv(
    os.path.join(feature_count_dir, "feature_count_summary.csv"))

print(f"\nFeature count plot and CSVs saved to: {feature_count_dir}")

In [ ]:
# Detect column types automatically from the training DataFrame
X_train_df: pandas.DataFrame = df_train.drop(columns=[TARGET_COLUMN])
y_train_series: pandas.Series = df_train[TARGET_COLUMN]

continuous_cols: list[str] = get_continuous_columns(X_train_df)
dummy_cols: list[str] = get_dummy_columns(X_train_df)

print(f"Continuous features: {len(continuous_cols)}")
print(f"Dummy features:     {len(dummy_cols)}")